# 05 - Train Logistic Regression and XGBoost

baseline logistic regression vs xgboost for high-cost / high-utilization risk classification.

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report,
)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier

with open("../configs/model_config.yaml") as f:
    config = yaml.safe_load(f)

LABEL = "high_cost"  # or "high_utilization"

FEATURE_COLS = [
    "inpatient_admits", "outpatient_visits", "carrier_claims",
    "inpatient_paid", "outpatient_paid", "carrier_paid",
    "has_alzheimers", "has_chf", "has_ckd", "has_cancer", "has_copd",
    "has_depression", "has_diabetes", "has_ihd", "has_osteoporosis",
    "has_ra_oa", "has_stroke_tia", "has_esrd",
    "age", "is_male", "race_black", "race_other", "race_hispanic",
    "rx_fill_count", "rx_total_cost", "rx_patient_paid",
    "other_payer_paid",
]

# the eda notebook (04) showed these are all heavily right-skewed dollar/
# count columns -- log1p-transforming them helps a linear model (logistic
# regression) but does nothing for tree models, since tree splits are
# invariant to any monotonic transform. only applied in the logreg pipeline
# below, never to xgboost/lightgbm's input.
SKEWED_COLS = [
    "inpatient_paid", "outpatient_paid", "carrier_paid",
    "outpatient_visits", "carrier_claims",
    "rx_fill_count", "rx_total_cost", "rx_patient_paid",
    "other_payer_paid",
]


**freezing the high_cost threshold from training data only.** `features_labeled.parquet` carries a *population-level* `high_cost` flag (top 10% of the full 52,375-member cohort) -- useful for the EDA notebook's descriptive numbers, but not for training a classifier: computing that 90th-percentile threshold from the full cohort lets the test set's own spend distribution quietly influence where the label boundary falls. The fix is to split first on the *continuous* 2010 spend, derive the dollar threshold only from the training rows, then apply that one frozen number to both train and test. `high_utilization` doesn't have this problem -- its threshold (2+ admits) is a fixed clinical rule set in `configs/model_config.yaml`, not something estimated from the data.

In [ ]:
# load the labeled feature table and split into train/test
df = pd.read_parquet("../data/processed/features_labeled.parquet")
X = df[FEATURE_COLS]

if LABEL == "high_cost":
    spend = df["total_paid_2010"]
    X_train, X_test, spend_train, spend_test = train_test_split(
        X, spend,
        test_size=config["split"]["test_size"],
        stratify=df["high_cost"] if config["split"]["stratify"] else None,
        random_state=config["random_seed"],
    )
    # the ONLY place this threshold is computed -- from spend_train alone
    cost_threshold = spend_train.quantile(config["label"]["high_cost_percentile"])
    y_train = (spend_train >= cost_threshold).astype(int)
    y_test = (spend_test >= cost_threshold).astype(int)
    print(f"frozen high_cost threshold (training data only): ${cost_threshold:,.0f}")
else:
    y = df[LABEL]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=config["split"]["test_size"],
        stratify=y if config["split"]["stratify"] else None,
        random_state=config["random_seed"],
    )


In [ ]:
# small helper to score a fitted classifier on the held-out test set
def evaluate(model, X_test, y_test):
    proba = model.predict_proba(X_test)[:, 1]
    return {
        "auc": round(roc_auc_score(y_test, proba), 4),
        "average_precision": round(average_precision_score(y_test, proba), 4),
        "brier_score": round(brier_score_loss(y_test, proba), 4),
    }


In [ ]:
# train the logistic regression baseline. skewed dollar/count columns get a
# log1p transform first (see SKEWED_COLS above); the rest -- chronic
# condition flags, demographics, inpatient_admits -- pass through unchanged
lr_config = config["logistic_regression"]

# the eda notebook also found a handful of rows with NEGATIVE payment
# amounts (real de-synpuf claim-adjustment artifacts) -- log1p of a value
# below -1 is undefined (nan), so those are clipped to 0 before the log,
# same as treating a claim reversal as "no net spend" for this transform only
def log1p_clipped(x):
    return np.log1p(np.clip(x, a_min=0, a_max=None))

log_transform = ColumnTransformer(
    [("log1p", FunctionTransformer(log1p_clipped), SKEWED_COLS)],
    remainder="passthrough",
)

logreg_pipeline = Pipeline([
    ("log_transform", log_transform),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        penalty=lr_config["penalty"],
        C=lr_config["C"],
        max_iter=lr_config["max_iter"],
        class_weight=lr_config["class_weight"],
    )),
])
logreg_pipeline.fit(X_train, y_train)

logreg_metrics = evaluate(logreg_pipeline, X_test, y_test)
logreg_metrics


In [ ]:
# xgboost needs its own imbalance handling -- logistic regression gets
# class_weight="balanced" from the config, but xgboost's equivalent knob,
# scale_pos_weight, isn't set anywhere by default. for a severely imbalanced
# label like high_utilization (~1.6% positive, a ~62:1 ratio) that matters a
# lot; for high_cost (~10% positive) it matters less but still helps
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight for {LABEL}: {scale_pos_weight:.1f}")


**why early stopping, not just a fixed `n_estimators`.** an ablation (outside this notebook) tested every obvious lever against the train/test AUC gap: removing `scale_pos_weight` entirely barely moved it (0.072 -> 0.061), and so did adding L1/L2 regularization (-> ~0.065). what actually mattered was plain tree capacity -- `max_depth=3` alone cut the gap in half, and early stopping (halting boosting once a held-out validation slice stops improving) did best of all: smaller gap **and** slightly higher test AUC than the fixed-300-round baseline. that's the tell that the extra rounds were fitting training-specific noise, not real signal, so the fix here is to let the data pick the round count once, rather than fixing `n_estimators=300` by hand.

In [ ]:
# carve a small validation slice out of the training partition, purely to
# decide how many boosting rounds to use -- never touches X_test/y_test
X_fit, X_earlystop_val, y_fit, y_earlystop_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=config["random_seed"]
)


In [ ]:
# train xgboost for nonlinear risk prediction, with early stopping picking
# the round count instead of a hand-fixed n_estimators
xgb_config = config["xgboost"]
xgb_search = XGBClassifier(
    n_estimators=xgb_config["n_estimators"],  # upper bound on rounds, not the final count
    max_depth=xgb_config["max_depth"],
    learning_rate=xgb_config["learning_rate"],
    subsample=xgb_config["subsample"],
    colsample_bytree=xgb_config["colsample_bytree"],
    eval_metric=xgb_config["eval_metric"],
    scale_pos_weight=scale_pos_weight,
    random_state=config["random_seed"],
    early_stopping_rounds=20,
)
xgb_search.fit(X_fit, y_fit, eval_set=[(X_earlystop_val, y_earlystop_val)], verbose=False)
print(f"xgboost early stopping selected {xgb_search.best_iteration} rounds (of {xgb_config['n_estimators']} max)")

# refit on the FULL training partition at that fixed round count -- early
# stopping's job was only to pick this number, not to be the final model
xgb_model = XGBClassifier(
    n_estimators=xgb_search.best_iteration,
    max_depth=xgb_config["max_depth"],
    learning_rate=xgb_config["learning_rate"],
    subsample=xgb_config["subsample"],
    colsample_bytree=xgb_config["colsample_bytree"],
    eval_metric=xgb_config["eval_metric"],
    scale_pos_weight=scale_pos_weight,
    random_state=config["random_seed"],
)
xgb_model.fit(X_train, y_train)

xgb_metrics = evaluate(xgb_model, X_test, y_test)
xgb_metrics


In [ ]:
# lightgbm: a second gradient-boosted-tree model, faster to train than
# xgboost on larger data and often competitive or better on tabular data.
# same scale_pos_weight and early-stopping treatment as xgboost, so it's a
# direct apples-to-apples comparison rather than a differently-tuned model
lgbm_config = config["lightgbm"]
#  lightgbm registers its own default eval metric (binary_logloss, from the
# classifier's objective) alongside whatever's passed to fit()'s
# eval_metric -- early stopping then tracks THAT one, not auc, unless
# metric="auc" is set on the constructor itself. binary_logloss gets worse
# every round on this data (the model is trading calibration for ranking as
# scale_pos_weight reweights it), so without this fix early stopping picked
# round 1 as "best" every time -- a real bug caught by inspecting the
# per-round trajectory, not something to trust silently.
lgbm_search = LGBMClassifier(
    n_estimators=lgbm_config["n_estimators"],
    max_depth=lgbm_config["max_depth"],
    num_leaves=lgbm_config["num_leaves"],
    learning_rate=lgbm_config["learning_rate"],
    subsample=lgbm_config["subsample"],
    colsample_bytree=lgbm_config["colsample_bytree"],
    scale_pos_weight=scale_pos_weight,
    random_state=config["random_seed"],
    verbose=-1,
    metric="auc",
)
lgbm_search.fit(
    X_fit, y_fit, eval_set=[(X_earlystop_val, y_earlystop_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
)
print(f"lightgbm early stopping selected {lgbm_search.best_iteration_} rounds (of {lgbm_config['n_estimators']} max)")

lgbm_model = LGBMClassifier(
    n_estimators=lgbm_search.best_iteration_,
    max_depth=lgbm_config["max_depth"],
    num_leaves=lgbm_config["num_leaves"],
    learning_rate=lgbm_config["learning_rate"],
    subsample=lgbm_config["subsample"],
    colsample_bytree=lgbm_config["colsample_bytree"],
    scale_pos_weight=scale_pos_weight,
    random_state=config["random_seed"],
    verbose=-1,
)
lgbm_model.fit(X_train, y_train)

lgbm_metrics = evaluate(lgbm_model, X_test, y_test)
lgbm_metrics


**overfitting check: train vs. test.** every metric so far has only been computed on the test set -- which tells you how good the model is, but not whether it's *memorizing* the training data rather than generalizing. the direct check is to score each model on the data it was fit on too: a large gap (e.g. train AUC 0.95, test AUC 0.72) means the model has fit noise specific to the training rows; a small gap means it's learned a pattern that holds up on unseen data.

In [ ]:
train_vs_test = pd.DataFrame({
    "logistic_regression": {**{f"train_{k}": v for k, v in evaluate(logreg_pipeline, X_train, y_train).items()},
                             **{f"test_{k}": v for k, v in logreg_metrics.items()}},
    "xgboost": {**{f"train_{k}": v for k, v in evaluate(xgb_model, X_train, y_train).items()},
                **{f"test_{k}": v for k, v in xgb_metrics.items()}},
    "lightgbm": {**{f"train_{k}": v for k, v in evaluate(lgbm_model, X_train, y_train).items()},
                 **{f"test_{k}": v for k, v in lgbm_metrics.items()}},
}).T

train_vs_test["auc_gap"] = train_vs_test["train_auc"] - train_vs_test["test_auc"]
train_vs_test["ap_gap"] = train_vs_test["train_average_precision"] - train_vs_test["test_average_precision"]
train_vs_test[["train_auc", "test_auc", "auc_gap", "train_average_precision", "test_average_precision", "ap_gap"]]


**A single train/test split is noisy.** Re-running this notebook end to end can move the reported AUC by ~0.02 just from a different random partition landing on the same data -- not because anything about the model changed. 5-fold cross-validation below refits each model 5 times and reports the mean +/- spread, which is the number actually worth trusting/reporting, not the single split above. Crucially, this runs **only on the training partition** (`X_train`/`y_train`), never the full dataset -- the held-out test set stays untouched for every fold, exactly like the frozen `high_cost` threshold above: split once, do all tuning/CV inside training, evaluate once on data the model (and the label-freezing) never saw.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=config["random_seed"])
scoring = ["roc_auc", "average_precision"]

# cross_validate clones each estimator fresh per fold, so passing the
# already-fit pipeline/model objects is fine -- their hyperparameters
# (including scale_pos_weight, fixed from the full training set) carry over,
# only the fit itself is redone per fold. run on X_train/y_train ONLY, not
# the full X/y -- X_test/y_test is reserved entirely for the one-shot
# evaluation later in this notebook
logreg_cv = cross_validate(logreg_pipeline, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)
xgb_cv = cross_validate(xgb_model, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)
lgbm_cv = cross_validate(lgbm_model, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)

cv_summary = pd.DataFrame({
    "logistic_regression": [
        logreg_cv["test_roc_auc"].mean(), logreg_cv["test_roc_auc"].std(),
        logreg_cv["test_average_precision"].mean(), logreg_cv["test_average_precision"].std(),
    ],
    "xgboost": [
        xgb_cv["test_roc_auc"].mean(), xgb_cv["test_roc_auc"].std(),
        xgb_cv["test_average_precision"].mean(), xgb_cv["test_average_precision"].std(),
    ],
    "lightgbm": [
        lgbm_cv["test_roc_auc"].mean(), lgbm_cv["test_roc_auc"].std(),
        lgbm_cv["test_average_precision"].mean(), lgbm_cv["test_average_precision"].std(),
    ],
}, index=["auc_mean", "auc_std", "ap_mean", "ap_std"])
cv_summary


**overfitting check, cross-validated version.** `return_train_score=True` above also captured each fold's *training* score, not just its validation score -- so the train-vs-test gap can be checked 5 times over (once per fold) instead of relying on the single split above. a consistent gap across folds is a much stronger signal than the single-split gap alone, since it rules out the gap being a fluke of one particular partition.

In [ ]:
cv_overfit_summary = pd.DataFrame({
    "logistic_regression": [logreg_cv["train_roc_auc"].mean(), logreg_cv["test_roc_auc"].mean(),
                             logreg_cv["train_roc_auc"].mean() - logreg_cv["test_roc_auc"].mean()],
    "xgboost": [xgb_cv["train_roc_auc"].mean(), xgb_cv["test_roc_auc"].mean(),
                xgb_cv["train_roc_auc"].mean() - xgb_cv["test_roc_auc"].mean()],
    "lightgbm": [lgbm_cv["train_roc_auc"].mean(), lgbm_cv["test_roc_auc"].mean(),
                 lgbm_cv["train_roc_auc"].mean() - lgbm_cv["test_roc_auc"].mean()],
}, index=["train_auc_mean", "validation_auc_mean", "auc_gap"])
cv_overfit_summary


**Validating the predictions further.** AUC/AP on their own -- single split or cross-validated -- are still only about ranking quality. Two more things matter for deciding whether this model is actually usable: are the predicted probabilities *calibrated* (does "70% risk" really mean 70% of those members turned out positive?), and what happens at the threshold you'd actually operate at.

In [ ]:
# calibration / reliability curve: bucket predictions into deciles and
# compare the average predicted probability to the actual positive rate in
# that bucket. a well-calibrated model's points sit on the diagonal
logreg_proba = logreg_pipeline.predict_proba(X_test)[:, 1]
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
lgbm_proba = lgbm_model.predict_proba(X_test)[:, 1]

logreg_frac_pos, logreg_mean_pred = calibration_curve(y_test, logreg_proba, n_bins=10, strategy="quantile")
xgb_frac_pos, xgb_mean_pred = calibration_curve(y_test, xgb_proba, n_bins=10, strategy="quantile")
lgbm_frac_pos, lgbm_mean_pred = calibration_curve(y_test, lgbm_proba, n_bins=10, strategy="quantile")

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="perfectly calibrated")
ax.plot(logreg_mean_pred, logreg_frac_pos, marker="o", label="logistic regression")
ax.plot(xgb_mean_pred, xgb_frac_pos, marker="o", label="xgboost")
ax.plot(lgbm_mean_pred, lgbm_frac_pos, marker="o", label="lightgbm")
ax.set_xlabel("mean predicted probability (decile bucket)")
ax.set_ylabel("observed positive rate")
ax.set_title(f"calibration -- {LABEL}")
ax.legend()
plt.show()


In [ ]:
# confusion matrix + precision/recall at an actual operating threshold.
# a fixed 0.5 cutoff is a poor default under heavy imbalance -- with only
# ~1.6% positive for high_utilization, almost nothing clears 0.5. a payer
# would instead pick a threshold that flags a workable share of the cohort
# (e.g. "the top 10% by risk score gets outreach"), so both are shown here
OPERATING_PERCENTILE = 0.90  # flag the top 10% of predicted risk

threshold_fixed = 0.5
threshold_percentile = pd.Series(xgb_proba).quantile(OPERATING_PERCENTILE)
thresholds = [("fixed 0.5 cutoff", threshold_fixed), (f"top {(1 - OPERATING_PERCENTILE):.0%} by risk score", threshold_percentile)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, (name, threshold) in zip(axes, thresholds):
    preds = (xgb_proba >= threshold).astype(int)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax,
        xticklabels=[f"predicted\nnot {LABEL}", f"predicted\n{LABEL}"],
        yticklabels=[f"actually\nnot {LABEL}", f"actually\n{LABEL}"],
        annot_kws={"size": 12},
    )
    ax.set_title(f"xgboost -- {name}\n(threshold={threshold:.3f})", fontsize=10)
    print(f"--- xgboost, {name} (threshold={threshold:.3f}) ---")
    print(classification_report(y_test, preds, digits=3, zero_division=0))
plt.tight_layout()
plt.show()


In [ ]:
# save all three models + metrics for the shap and dashboard steps
out_dir = Path("../data/processed/models")
out_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(logreg_pipeline, out_dir / f"logistic_regression_{LABEL}.joblib")
joblib.dump(xgb_model, out_dir / f"xgboost_{LABEL}.joblib")
joblib.dump(lgbm_model, out_dir / f"lightgbm_{LABEL}.joblib")

with open(out_dir / f"logistic_regression_{LABEL}_metrics.json", "w") as f:
    json.dump(logreg_metrics, f, indent=2)
with open(out_dir / f"xgboost_{LABEL}_metrics.json", "w") as f:
    json.dump(xgb_metrics, f, indent=2)
with open(out_dir / f"lightgbm_{LABEL}_metrics.json", "w") as f:
    json.dump(lgbm_metrics, f, indent=2)

# the frozen training-only threshold, so notebook 08 (targeting) can derive
# the same high_cost definition for the whole cohort instead of reusing the
# population-level column -- keeps every downstream use of "high_cost"
# consistent with what the model was actually trained to predict
if LABEL == "high_cost":
    with open(out_dir / "high_cost_threshold.json", "w") as f:
        json.dump({"cost_threshold": float(cost_threshold)}, f, indent=2)

pd.DataFrame(
    [logreg_metrics, xgb_metrics, lgbm_metrics],
    index=["logistic_regression", "xgboost", "lightgbm"],
)
